In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 138.3 MB/s eta 0:00:00


In [2]:
%%time
%%capture
!uv pip install vllm --torch-backend=auto --system
# !uv pip install vllm-flash-attn --torch-backend=auto --system

CPU times: user 28.1 ms, sys: 8.64 ms, total: 36.7 ms
Wall time: 42.4 s


In [3]:
%%time
%%capture
!pip install torch-c-dlpack-ext

CPU times: user 2.84 ms, sys: 2.14 ms, total: 4.99 ms
Wall time: 1.02 s


In [4]:
!pip show vllm

Name: vllm
Version: 0.15.1
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /opt/conda/lib/python3.12/site-packages
Requires: aiohttp, anthropic, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, flashinfer-python, gguf, grpcio, grpcio-reflection, ijson, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xgr

In [5]:
%%time
import pandas as pd
from huggingface_hub import hf_hub_download
from tqdm import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

CPU times: user 9.17 s, sys: 2.27 s, total: 11.4 s
Wall time: 8.88 s


In [7]:
print("=" * 60)
print("Loading data ...")
print("=" * 60)

repo_id = "aatman/search-activity"
file_path = "chatgpt/conversations.csv"

cached_file = hf_hub_download(
    repo_id=repo_id,
    filename=file_path,
    repo_type="dataset",
)
print(f"File cached at: {cached_file}")

Loading data ...


chatgpt/conversations.csv:   0%|          | 0.00/491M [00:00<?, ?B/s]

File cached at: /home/jovyan/.cache/huggingface/hub/datasets--aatman--search-activity/snapshots/af9c4b9a63081b5ab3b329b16ae2e563709e3dfd/chatgpt/conversations.csv


In [8]:
%%time
df = pd.read_csv(cached_file, low_memory=False)
print(f"Loaded dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Loaded dataframe shape: (213674, 14)
Columns: ['user_id', 'internal_user_id', 'chatgpt_plus_user', 'birth_year', 'conversation_id', 'conversation_title', 'conversation_create_time', 'conversation_update_time', 'system_instruction', 'prompt', 'response', 'prompt_create_time', 'response_create_time', 'response_model']
CPU times: user 2.61 s, sys: 223 ms, total: 2.83 s
Wall time: 2.84 s


In [9]:
df.head()

,user_id,internal_user_id,chatgpt_plus_user,birth_year,conversation_id,conversation_title,conversation_create_time,conversation_update_time,system_instruction,prompt,response,prompt_create_time,response_create_time,response_model
0,20529737_1,user-kZHjyl6HEAsYnEUnDDnoLNG6,False,1991,69660ae2-c408-8322-a7ab-a930a3829e13,Predicting Personal Attributes,1.768295e+09,1.768296e+09,NaN,"based on your interactions with me, can you pr...",Sure. I’ll narrow it down **by region (West Be...,1.768295e+09,1.768296e+09,gpt-5-2
1,20529737_1,user-kZHjyl6HEAsYnEUnDDnoLNG6,False,1991,69660ae2-c408-8322-a7ab-a930a3829e13,Predicting Personal Attributes,1.768295e+09,1.768296e+09,NaN,What is my ethnicity or cultural background?,Sure. I’ll narrow it down **by region (West Be...,1.768296e+09,1.768296e+09,gpt-5-2
2,20529737_1,user-kZHjyl6HEAsYnEUnDDnoLNG6,False,1991,69660ae2-c408-8322-a7ab-a930a3829e13,Predicting Personal Attributes,1.768295e+09,1.768296e+09,NaN,Can you suggest cultural influences based on g...,Sure. I’ll narrow it down **by region (West Be...,1.768296e+09,1.768296e+09,gpt-5-2
3,20529737_1,user-kZHjyl6HEAsYnEUnDDnoLNG6,False,1991,69660ae2-c408-8322-a7ab-a930a3829e13,Predicting Personal Attributes,1.768295e+09,1.768296e+09,NaN,Based on the previous response from ChatGPT ab...,Sure. I’ll narrow it down **by region (West Be...,1.768296e+09,1.768296e+09,gpt-5-2
4,20529737_1,user-kZHjyl6HEAsYnEUnDDnoLNG6,False,1991,69660ae2-c408-8322-a7ab-a930a3829e13,Predicting Personal Attributes,1.768295e+09,1.768296e+09,NaN,Based on the previous response from ChatGPT ab...,Sure — I’ll narrow it down **by region: West B...,1.768296e+09,1.768296e+09,gpt-5-2


In [10]:
df["conversation_id"].nunique()

41054

In [11]:
def build_conversation_text(group: pd.DataFrame) -> str:
    """Concatenate all prompt/response turns chronologically."""
    group = group.sort_values("prompt_create_time", na_position="last")

    turns = []
    for _, row in group.iterrows():
        prompt = str(row["prompt"]).strip() if pd.notna(row["prompt"]) else ""
        response = str(row["response"]).strip() if pd.notna(row["response"]) else ""

        if prompt:
            turns.append(f"User: {prompt}")

        if response:
            if len(response) > 1000:
                response = response[:1000] + " [...]"
            turns.append(f"Assistant: {response}")

    return "\n\n".join(turns)

In [12]:
MAX_CHARS = 25_000  # adjust based on your GPU/context


def safe_truncate(text: str) -> str:
    if len(text) > MAX_CHARS:
        return text[:MAX_CHARS] + "\n\n[Conversation truncated]"
    return text

In [13]:
def build_messages(conversation_text: str):
    """Build chat messages for Qwen3 summarization."""
    conversation_text = safe_truncate(conversation_text)

    return [
        {
            "role": "system",
            "content": (
                "You are a precise assistant that summarizes conversations.\n"
                "Write clear, factual summaries."
            ),
        },
        {
            "role": "user",
            "content": (
                "Summarize the following conversation between a user and an assistant.\n\n"
                "Requirements:\n"
                "- Write 2–3 sentences.\n"
                "- Focus on the main goal and key outcomes.\n"
                "- Be concise and factual.\n"
                "- Do NOT include reasoning.\n"
                "- Do NOT copy text verbatim.\n\n"
                "Conversation:\n"
                f"{conversation_text}\n\n"
                "Summary:"
            ),
        },
    ]

In [14]:
print("\nGrouping conversations ...")
conv_groups = df.groupby("conversation_id")

records = []
for conv_id, group in tqdm(conv_groups, desc="Building conversation texts"):
    user_id = group["user_id"].iloc[0]
    conv_text = build_conversation_text(group)
    n_turns = len(group)

    records.append(
        {
            "user_id": user_id,
            "conversation_id": conv_id,
            "conversation_text": conv_text,
            "n_turns": n_turns,
        }
    )


Grouping conversations ...


Building conversation texts: 100%|██████████| 41054/41054 [00:10<00:00, 3966.52it/s]


In [15]:
conv_df = pd.DataFrame(records)
print(f"\nUnique conversations: {len(conv_df):,}")
print(conv_df["n_turns"].describe())


Unique conversations: 41,054
count    41054.000000
mean         5.204706
std         12.775843
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max       1160.000000
Name: n_turns, dtype: float64


In [16]:
model_name = "Qwen/Qwen3-8B"

In [17]:
%%time
llm = LLM(
    model=model_name,
    dtype="auto",
    gpu_memory_utilization=0.95,
)

INFO 02-22 13:18:37 [utils.py:261] non-default args: {'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

INFO 02-22 13:18:45 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-22 13:18:45 [model.py:1561] Using max model len 40960
INFO 02-22 13:18:45 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-22 13:18:45 [vllm.py:624] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

(EngineCore_DP0 pid=1231) INFO 02-22 13:19:21 [weight_utils.py:527] Time spent downloading weights for Qwen/Qwen3-8B: 31.488361 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1231) INFO 02-22 13:19:39 [default_loader.py:291] Loading weights took 17.61 seconds
(EngineCore_DP0 pid=1231) INFO 02-22 13:19:39 [gpu_model_runner.py:4130] Model loading took 15.27 GiB memory and 50.326547 seconds
(EngineCore_DP0 pid=1231) INFO 02-22 13:19:46 [backends.py:812] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/ebd61e1f2c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1231) INFO 02-22 13:19:46 [backends.py:872] Dynamo bytecode transform time: 6.00 s
(EngineCore_DP0 pid=1231) INFO 02-22 13:19:55 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=1231) INFO 02-22 13:20:00 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 11.92 s
(EngineCore_DP0 pid=1231) INFO 02-22 13:20:00 [monitor.py:34] torch.compile takes 17.92 s in total
(EngineCore_DP0 pid=1231) INFO 02-22 13:20:01 [gpu_worker.py:356] Available KV cache memory: 25.54 GiB
(EngineCore_DP0 pid=1231) IN

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 19.62it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 22.57it/s]


(EngineCore_DP0 pid=1231) INFO 02-22 13:20:06 [gpu_model_runner.py:5063] Graph capturing finished in 5 secs, took 0.60 GiB
(EngineCore_DP0 pid=1231) INFO 02-22 13:20:06 [core.py:272] init engine (profile, create kv cache, warmup model) took 26.82 seconds
INFO 02-22 13:20:08 [llm.py:343] Supported tasks: ['generate']
CPU times: user 2.01 s, sys: 381 ms, total: 2.39 s
Wall time: 1min 31s


In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=0.9,
    max_tokens=120,
    repetition_penalty=1.05,
)


def format_for_qwen(messages):
    """Apply Qwen chat template with thinking disabled."""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

In [19]:
print("\n" + "=" * 60)
print("SANITY CHECK — shortest conversation")
print("=" * 60)

shortest = conv_df.sort_values("n_turns").iloc[0]

print(f"user_id        : {shortest['user_id']}")
print(f"conversation_id: {shortest['conversation_id']}")
print(f"n_turns        : {shortest['n_turns']}")

print("\nRunning sanity-check summary ...")

example_messages = build_messages(shortest["conversation_text"])
example_prompt = format_for_qwen(example_messages)
print("-------------------\n Prompt:")
print(example_prompt)
print()

example_output = llm.generate([example_prompt], sampling_params)
example_summary = example_output[0].outputs[0].text.strip()

print("\n--- LLM Summary (shortest conversation) ---")
print(example_summary)
print("=" * 60)


SANITY CHECK — shortest conversation
user_id        : 25417647
conversation_id: 01ec27c5-de3f-41a2-9975-e790de4657cd
n_turns        : 1

Running sanity-check summary ...
-------------------
 Prompt:
<|im_start|>system
You are a precise assistant that summarizes conversations.
Write clear, factual summaries.<|im_end|>
<|im_start|>user
Summarize the following conversation between a user and an assistant.

Requirements:
- Write 2–3 sentences.
- Focus on the main goal and key outcomes.
- Be concise and factual.
- Do NOT include reasoning.
- Do NOT copy text verbatim.

Conversation:
User: function getRemainingTime(date, time) {\n  \n    const targetDateTime = new Date(`${date} ${time}`);\n    const Currentdate = new Date();\n  \n    const timeDifference = targetDateTime - Currentdate;\n    let remainingTimeInSecs = Math.floor(timeDifference / 1000);\n     return remainingTimeInSecs;\n    \n  }\nstop the function when the value is less then or equal to 0 and reload the window

Assistant: To

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


--- LLM Summary (shortest conversation) ---
The user asked for a JavaScript function to calculate remaining time until a specific date and time and to reload the window when the time is up. The assistant provided a modified function that checks if the remaining time is less than or equal to zero and reloads the window in that case.


In [20]:
%%time
print(f"\nBuilding prompts for all {len(conv_df):,} conversations ...")

all_prompts = []
for _, row in conv_df.iterrows():
    msgs = build_messages(row["conversation_text"])
    formatted = format_for_qwen(msgs)
    all_prompts.append(formatted)


Building prompts for all 41,054 conversations ...
CPU times: user 3.68 s, sys: 308 ms, total: 3.99 s
Wall time: 4 s


In [21]:
%%time
print("Running batch inference ...")
outputs = llm.generate(all_prompts, sampling_params)

summaries = [out.outputs[0].text.strip() for out in outputs]
conv_df["llm_summary"] = summaries

Running batch inference ...


Adding requests:   0%|          | 0/41054 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/41054 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

CPU times: user 1min 53s, sys: 3.61 s, total: 1min 56s
Wall time: 1h 56min 20s


In [22]:
output_path = "conversation_summaries.csv"
conv_df[["user_id", "conversation_id", "llm_summary"]].to_csv(output_path, index=False)

print(f"\nSummaries saved to: {output_path}")
print(f"Total rows       : {len(conv_df):,}")


Summaries saved to: conversation_summaries.csv
Total rows       : 41,054


In [23]:
print("\nSample output:")
print(conv_df[["user_id", "conversation_id", "llm_summary"]].head(3).to_string())


Sample output:
    user_id                       conversation_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                   llm_summary
0  15533591  000184da-4ff8-4d59-9795-627d49ca4658                                                                                The user repeatedly attempted to deploy an NFT subgraph using the goldsky tool but encountered errors indicating the ABI file lacked events or functions. The main issue was an incomplete or invalid ABI file, which prevented successful subgraph generation. The assistant provided troubleshooting steps to verify the ABI content, file validity, 